# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedsamymohamad/flyrank_internship_starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

**Lane 2 — Refresh / Content Opportunity Scoring**

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Binary classification used as a scorer, output consumed as a ranked queue.**

Lane 2 asks: *which pages in a client's content inventory should an editor review first?* The output is a **priority score** (the model's estimated probability of decline) attached to every page, which an editor reads top-to-bottom. That makes it a **ranking / scoring** problem in practice, built on top of a binary classifier.

Why not pure ranking (e.g. a learning-to-rank model)? Because the starter dataset gives us a well-defined binary label (`is_declining_label`) that can be used directly as a supervised target. The ranked queue is then just `sort_values(score, ascending=False)` on the classifier's output probabilities. This is the most honest framing given the available label.

Why not clustering? Clustering is unsupervised and produces groups — it answers "what kinds of pages exist?", not "which ones need attention first?" The editorial decision requires a priority order, not a taxonomy.

**One-paragraph frame (from the framing skill):**

> For **content editors and SEO strategists at FlyRank clients**, deciding **which pages to review for refresh or optimisation in the current sprint**, we will build a **scored and ranked action queue** from the starter dataset (30 000 rows, 32 pseudonymised clients), predicting/scoring **the probability that a page is currently declining in search impressions** (label: `is_declining_label = 1` when `trend_direction == "down"`), measured by **Precision@K on a client-holdout split**. A wrong call costs ~30–60 min of wasted editor time per false positive, or a continuing traffic loss per false negative. A plain rule isn't enough because the relevant signals — impressions volume, average position, CTR, content age, engagement rate, word count — interact in ways too tangled to capture with a single threshold. We will claim only **observational and decision-support** results: the ranking associates signals with outcomes; it does not prove that a refresh will cause recovery.

In [ ]:
# Confirm the task type and label are consistent with the data
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# The binary label the classifier will learn
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

print(f'Dataset shape       : {df.shape}')
print(f'Task type           : binary classification → ranked queue')
print(f'Label column        : is_declining_label')
print(f'Positive class (=1) : {df["is_declining_label"].sum():,} declining pages ({df["is_declining_label"].mean():.1%})')
print(f'Negative class (=0) : {(df["is_declining_label"]==0).sum():,} non-declining pages')

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target: `is_declining_label` — a rule-based proxy, not a directly observed outcome.**

The label is defined as:
```
is_declining_label = 1   when trend_direction == "down"
trend_direction    = "down"  when impressions_last_30d < impressions_prev_30d × 0.80
```

This is a **rule-based proxy**, not an observed editorial outcome (e.g. "editor reviewed this page and found it needed refreshing"). The rule is transparent and threshold-based: a 20% drop in impressions over the last 30 days vs the prior 30 days. That is a meaningful signal — a page losing a fifth of its search visibility in one month is a plausible candidate for editorial attention.

**What this means for claims:** I cannot claim the model predicts "true" decline (which would require a later-window measurement). I can claim it predicts whether a page *currently shows the declining-trend pattern* as defined by this rule. All results are stated as observational and decision-support, not causal.

**The leakage boundary:** `trend_direction` and `trend_pct` are derived from `impressions_last_30d` and `impressions_prev_30d`. Both are therefore **never features** — they are the label's parents. Using them would let the model read the answer directly. The starter script `scripts/01_prepare_features.py` drops them before fitting; this notebook does the same.

In [ ]:
# Show where the label comes from and confirm the leakage boundary
forbidden_features = ['trend_direction', 'trend_pct', 'content_id', 'client_id']

print('Label derivation chain:')
print('  impressions_last_30d + impressions_prev_30d')
print('      → trend_pct        = (last - prev) / prev × 100')
print('      → trend_direction  = "down" when trend_pct < -20')
print('      → is_declining_label = 1 when trend_direction == "down"')
print()

# Verify the label matches the rule manually (should be 100%)
manual_label = (
    df['impressions_prev_30d'].gt(0) &
    (df['impressions_last_30d'] < df['impressions_prev_30d'] * 0.80)
).astype(int)

match = (manual_label == df['is_declining_label']).mean()
print(f'Manual rule matches is_declining_label : {match:.1%}  ✓')
print()
print('Columns that must NEVER become model features (label sources or IDs):')
for c in forbidden_features:
    print(f'  ✗  {c}')

## 3. Success metric

*One metric you can defend. What number means "good"?*

**Primary metric: Precision@K (specifically Precision@50), evaluated on a client-holdout split.**

**Why Precision@K?** An editor reads the ranked queue from the top. If the top 50 pages are mostly false positives, the editor wastes time and loses trust in the tool. Precision@50 directly measures: of the 50 pages we surface first, what fraction are genuine declining pages?

**Why not accuracy?** With ~54% declining pages, a model that always predicts "declining" achieves 54% accuracy — useless. Accuracy rewards the majority class and tells us nothing about the top of the queue.

**Why not ROC-AUC alone?** AUC measures discrimination across all thresholds. Useful for comparing models, but an editor doesn't operate at all thresholds — they read the top K. Precision@K is the operationally honest metric.

**Baseline to beat:** A fixed rule achieves Precision@50 ≈ 0.24 on the full 30k dataset (from `outputs/model_report.md`). The Random Forest in the starter pipeline achieves ≈ 0.74. Any new model must beat 0.24 to be useful, evaluated on the same client-holdout split.

**What "good" means:** Precision@50 ≥ 0.60 on a client-holdout split with clients the model has never seen.

In [ ]:
# Compute Precision@K for known baselines and the project target
K = 50
base_rate    = df['is_declining_label'].mean()   # ~54%
rule_p_at_k  = 0.24   # from outputs/model_report.md (full 30k dataset)
rf_p_at_k    = 0.74   # from outputs/model_report.md (client-holdout)
target_p_at_k = 0.60  # project bar

print(f'Success metric : Precision@{K} on client-holdout split')
print()
print(f'  Label base rate (declining pages)  : {base_rate:.1%}')
print(f'  Naive baseline (always declining)  : ~{base_rate:.2f}  ({int(base_rate*K)}/{K} correct)')
print(f'  Rule-based baseline                : {rule_p_at_k:.2f}  ({int(rule_p_at_k*K)}/{K} correct)')
print(f'  Random Forest (starter pipeline)   : {rf_p_at_k:.2f}  ({int(rf_p_at_k*K)}/{K} correct)')
print()
print(f'  Project target bar                 : {target_p_at_k:.2f}  ({int(target_p_at_k*K)}/{K} correct)')
print(f'  Must beat {rule_p_at_k:.2f} to be useful over the rule baseline.')

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content item (page), aggregated over a trailing 90-day window, for one pseudonymised client.**

The unit of analysis is the **page-snapshot**: a single page's accumulated performance metrics (impressions, clicks, position, sessions, engagement) over the 90 days ending at export time. Each row is unique on `content_id`. The decision moment is "now" — the snapshot is the present state of the page. The label reflects what happened in the last 30 days vs the prior 30 days, both of which are inside the 90-day window.

This is **not a time series** at this stage — each row is a single observation per page. The warehouse release (weeks 3+) will allow us to build rolling features from the daily fact table, but the starter CSV is already pre-aggregated.

In [ ]:
# Show the unit of analysis as a real dataframe slice
# Lane 2 slice: pages with meaningful search visibility (impressions_90d >= 100)

lane_slice = df[df['impressions_90d'] >= 100].copy()

show_cols = [
    'content_id',            # context (ID — never a feature)
    'client_id',             # context (ID — never a feature)
    'content_type',          # feature candidate
    'impressions_90d',       # feature
    'avg_position',          # feature  (0 = no data, not rank zero)
    'ctr',                   # feature  (×100 pct — 0.76 means 0.76%)
    'days_with_impressions', # feature
    'content_age_days',      # feature
    'trend_direction',       # label source — EXCLUDED from features
    'is_declining_label',    # TARGET
]

print(f'Full dataset        : {len(df):,} rows  (one row = one page-snapshot)')
print(f'Lane 2 slice        : {len(lane_slice):,} rows  (impressions_90d >= 100)')
print(f'Unique clients      : {lane_slice["client_id"].nunique()}')
print(f'Declining (label=1) : {lane_slice["is_declining_label"].sum():,}  ({lane_slice["is_declining_label"].mean():.1%})')
print()
print('Sample rows — one row = one page-snapshot for one client:')
lane_slice[show_cols].head(5)

In [ ]:
# Show the target column distribution and key signal differences by label
print('Target column distribution (is_declining_label) in lane slice:')
print(
    lane_slice['is_declining_label']
    .value_counts()
    .rename({0: 'not declining (0)', 1: 'declining (1)'})
    .to_string()
)
print()

print('Median feature values by label (note: avg_position=0 means no data; ctr is ×100):')
signal_cols = ['impressions_90d', 'avg_position', 'ctr',
               'days_with_impressions', 'content_age_days']
print(
    lane_slice.groupby('is_declining_label')[signal_cols]
    .median()
    .rename(index={0: 'not declining', 1: 'declining'})
    .round(2)
    .to_string()
)

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Three reasons a fixed rule fails here:**

**1 — Multiple signals interact, and no single threshold captures them all.**
A declining page is not simply old, or low-impressions, or low-CTR — it is a specific *combination* of these. A page with 10 000 impressions and a CTR of 0.05% is a very different problem from one with 500 impressions and a CTR of 2%. Any rule that thresholds on one dimension ignores all others. The code cell below shows concretely how the declining rate varies across every combination of impression tier and position tier — there is no clean cut.

**2 — The pattern differs across clients.**
The dataset has 32 pseudonymised clients with different content strategies, keyword profiles, and traffic volumes. A rule calibrated on one client will be miscalibrated on another. A learned model adapts implicitly when evaluated on a client-holdout split.

**3 — The relevant signals are too numerous to threshold manually.**
The prepared feature vector has 40+ columns. No human can reason simultaneously about `days_with_impressions`, `log_impressions`, `avg_position`, `ctr`, `engagement_rate`, `scroll_rate`, `content_age_days`, and `word_count` for thousands of pages at once. A model does exactly this — and makes the weighting inspectable via feature importances.

In [ ]:
# Demonstrate the failure of a fixed rule at Precision@50
# Rule: flag pages with impressions_90d >= 500 AND content_age_days >= 180

rule_score = (
    (lane_slice['impressions_90d'] >= 500) &
    (lane_slice['content_age_days'] >= 180)
).astype(float)

top_k = lane_slice.copy()
top_k['rule_score'] = rule_score
top_k_sorted = top_k.sort_values('rule_score', ascending=False).head(50)
p_at_50_rule = top_k_sorted['is_declining_label'].mean()

print('Fixed rule: impressions_90d >= 500  AND  content_age_days >= 180')
print(f'  Pages flagged by the rule  : {int(rule_score.sum()):,}')
print(f'  Precision@50 (this rule)   : {p_at_50_rule:.2f}  ({int(p_at_50_rule*50)}/50 correct)')
print(f'  Precision@50 (RF baseline) : 0.74  (37/50 correct)  — from outputs/model_report.md')
print()
print(f'  The RF correctly surfaces {int((0.74 - p_at_50_rule)*50)} more declining pages')
print(f'  in the top 50 than this hand-crafted rule.')
print()
print('Why a rule fails: it thresholds on one or two dimensions.')
print('The declining pattern is a combination of position, volume, CTR, age, and engagement.')

In [ ]:
# Show that declining pages are spread across every combination of signals —
# there is no clean rule that separates them.

print('Declining rate by impression tier × position tier')
print('(each cell = fraction of pages in that cell that are declining):')
print()

pivot = (
    lane_slice
    .assign(
        imp_tier=pd.cut(
            lane_slice['impressions_90d'],
            bins=[0, 500, 2000, 10000, float('inf')],
            labels=['100–500', '500–2k', '2k–10k', '10k+']
        )
    )
    .groupby(['imp_tier', 'position_tier'], observed=True)['is_declining_label']
    .mean()
    .unstack('position_tier')
    .round(2)
)
print(pivot.to_string())
print()
print('Observation: declining pages appear in every cell — high AND low impression volume,')
print('top-3 AND deep positions. No single threshold cleanly separates declining from stable.')

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] The task type is named (binary classification → ranked queue)
- [x] The target/proxy is named and its leakage boundary is stated
- [x] The success metric (Precision@50, client-holdout) is defined before any training
- [x] One row = one page-snapshot is shown as a real dataframe with visible data
- [x] Why ML beats a fixed rule is backed by a concrete Precision@50 comparison
- [x] The output is tied to a real content action (editorial queue, refresh decision)
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.